# Exercise 1

In [ ]:
import torch
from torch.nn import functional as F

In [12]:
X = torch.tensor([[[[0,1,3], [2,3,5]],
                 [[1,2,4], [3,4,7]],
                 [[7,2,5], [3,5,8]]]], dtype=torch.float32)

K = torch.tensor([[[[0,1], [2,3]],
                 [[1,2], [3,4]],
                 [[7,2], [3,5]]]], dtype=torch.float32)

In [13]:
result = F.conv2d(X, K)

print("Input tensor shape:", X.shape)
print("Kernel shape:", K.shape)
print("Output tensor shape:", result.shape)
print("\nResult:")
print(result)

Input tensor shape: torch.Size([1, 3, 2, 3])
Kernel shape: torch.Size([1, 3, 2, 2])
Output tensor shape: torch.Size([1, 1, 1, 2])

Result:
tensor([[[[131., 153.]]]])


# Exercise 2

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

**Load SVHN dataset**

In [2]:
def load_data_svhn(batch_size):
    """Download the SVHN dataset and load it into memory."""
    trans = transforms.Compose([transforms.ToTensor()])
    
    # Load training and test datasets
    svhn_train_full = torchvision.datasets.SVHN(
        root="../data", split='train', transform=trans, download=True)
    svhn_test = torchvision.datasets.SVHN(
        root="../data", split='test', transform=trans, download=True)
    
    # Split training data: 30000 for training, 43257 for validation
    svhn_train, svhn_val = torch.utils.data.random_split(
        svhn_train_full, [30000, 43257],
        generator=torch.Generator().manual_seed(42))
    
    return (torch.utils.data.DataLoader(svhn_train, batch_size, shuffle=True, num_workers=2),
            torch.utils.data.DataLoader(svhn_val, batch_size, shuffle=False, num_workers=2),
            torch.utils.data.DataLoader(svhn_test, batch_size, shuffle=False, num_workers=2))


**Define LeNet model (adapted for 3 color channels and 32x32 images)**

In [3]:
net = nn.Sequential(
    nn.Conv2d(3, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 6 * 6, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10))

**Training functions**

In [ ]:
def evaluate_accuracy(net, data_iter, loss, device):
    """Compute the accuracy for a model on a dataset."""
    net.eval()
    total_loss = 0
    total_hits = 0
    total_samples = 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            total_loss += l.item()
            total_hits += sum(net(X).argmax(axis=1).type(y.dtype) == y)
            total_samples += y.numel()
    return float(total_loss) / len(data_iter), float(total_hits) / total_samples * 100

def train_epoch(net, train_iter, loss, optimizer, device):
    net.train()
    total_loss = 0
    total_hits = 0
    total_samples = 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        y_hat = net(X)
        l = loss(y_hat, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        total_loss += l.item()
        total_hits += sum(y_hat.argmax(axis=1).type(y.dtype) == y)
        total_samples += y.numel()
    return float(total_loss) / len(train_iter), float(total_hits) / total_samples * 100

def train(net, train_iter, val_iter, test_iter, num_epochs, lr, device):
    """Train a model."""
    train_loss_all = []
    train_acc_all = []
    val_loss_all = []
    val_acc_all = []
    
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    
    net.apply(init_weights)
    print('Training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(net, train_iter, loss, optimizer, device)
        train_loss_all.append(train_loss)
        train_acc_all.append(train_acc)
        val_loss, val_acc = evaluate_accuracy(net, val_iter, loss, device)
        val_loss_all.append(val_loss)
        val_acc_all.append(val_acc)
        print(f'Epoch {epoch + 1}, Train loss {train_loss:.2f}, Train accuracy {train_acc:.2f}, Validation loss {val_loss:.2f}, Validation accuracy {val_acc:.2f}')
    
    test_loss, test_acc = evaluate_accuracy(net, test_iter, loss, device)
    print(f'Test loss {test_loss:.2f}, Test accuracy {test_acc:.2f}')
    
    return train_loss_all, train_acc_all, val_loss_all, val_acc_all

def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

**Plots**

In [5]:
def plot_loss(train_loss_all, val_loss_all):
    epochs = range(1, len(train_loss_all) + 1)
    plt.plot(epochs, train_loss_all, 'bo', label='Training loss')
    plt.plot(epochs, val_loss_all, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

def plot_accuracy(train_acc_all, val_acc_all):
    epochs = range(1, len(train_acc_all) + 1)
    plt.plot(epochs, train_acc_all, 'bo', label='Training acc')
    plt.plot(epochs, val_acc_all, 'b', label='Validation acc')
    plt.title('Training and validation accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

**Train model and plot results**

In [9]:
batch_size, lr, num_epochs = 256, 0.5, 10
train_iter, val_iter, test_iter = load_data_svhn(batch_size)
train_loss_all, train_acc_all, val_loss_all, val_acc_all = train(net, train_iter, val_iter, test_iter, num_epochs, lr, try_gpu())

plot_loss(train_loss_all, val_loss_all)
plot_accuracy(train_acc_all, val_acc_all)

Training on cpu
Epoch 1, Train loss 2.27, Train accuracy 16.90, Validation loss 2.28, Validation accuracy 14.31
Epoch 2, Train loss 2.25, Train accuracy 18.24, Validation loss 2.26, Validation accuracy 18.97
Epoch 3, Train loss 2.24, Train accuracy 18.32, Validation loss 2.24, Validation accuracy 18.97
Epoch 4, Train loss 2.24, Train accuracy 18.71, Validation loss 2.24, Validation accuracy 18.97
Epoch 5, Train loss 2.24, Train accuracy 18.85, Validation loss 2.27, Validation accuracy 11.54


KeyboardInterrupt: 

# Exercise 3

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

**Convolutional block class**

In [ ]:
class ConvBlock(nn.Module):
    """
    Convolutional Block with residual connection.
    
    Architecture (following the diagram):
    - Left path: Conv2d(64, 3×3) → ReLU
    - Right path: Conv2d(32, 3×3) → Conv2d(64, 3×3) → BatchNorm2d → ReLU
    - Output: Add both paths
    """
    
    def __init__(self, in_channels):
        super(ConvBlock, self).__init__()
        
        # Left path (skip/shortcut connection)
        self.conv_left = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.relu_left = nn.ReLU()
        
        # Right path (main processing path)
        self.conv1_right = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2_right = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn_right = nn.BatchNorm2d(64)
        self.relu_right = nn.ReLU()
    
    def forward(self, x):
        # Left path (shortcut)
        left = self.conv_left(x)
        left = self.relu_left(left)
        
        # Right path (main)
        right = self.conv1_right(x)
        right = self.conv2_right(right)
        right = self.bn_right(right)
        right = self.relu_right(right)
        
        # Add both paths element-wise
        output = left + right
        
        return output

**Complete network**

In [ ]:
class CustomCNN(nn.Module):
    """
    Custom CNN with Convolutional Block.
    
    Full architecture:
    Input (3, 32, 32) → ConvBlock → Flatten → Linear(10)
    """
    
    def __init__(self):
        super(CustomCNN, self).__init__()
        
        # Convolutional block (input: 3 channels, output: 64 channels)
        self.conv_block = ConvBlock(in_channels=3)
        
        # Flatten layer
        self.flatten = nn.Flatten()
        
        # Fully connected layer
        # After ConvBlock: 64 channels × 32 × 32 = 65536
        self.fc = nn.Linear(64 * 32 * 32, 10)
    
    def forward(self, x):
        x = self.conv_block(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

# Create the network
net = CustomCNN()

# Print network architecture
print("Network Architecture:")
print(net)
print("\n" + "="*70)

# Test with a sample input to verify dimensions
sample_input = torch.randn(1, 3, 32, 32)
output = net(sample_input)
print(f"\nInput shape: {sample_input.shape}")
print(f"Output shape: {output.shape}")
print("="*70 + "\n")

**Load data**

In [ ]:
def load_data_svhn(batch_size):
    """Download the SVHN dataset and load it into memory."""
    trans = transforms.Compose([transforms.ToTensor()])
    
    svhn_train_full = torchvision.datasets.SVHN(
        root="../data", split='train', transform=trans, download=True)
    svhn_test = torchvision.datasets.SVHN(
        root="../data", split='test', transform=trans, download=True)
    
    # Split training data: 30000 for training, 43257 for validation
    svhn_train, svhn_val = torch.utils.data.random_split(
        svhn_train_full, [30000, 43257],
        generator=torch.Generator().manual_seed(42))
    
    return (torch.utils.data.DataLoader(svhn_train, batch_size, shuffle=True, num_workers=2),
            torch.utils.data.DataLoader(svhn_val, batch_size, shuffle=False, num_workers=2),
            torch.utils.data.DataLoader(svhn_test, batch_size, shuffle=False, num_workers=2))


**Training functions**

In [ ]:
def evaluate_accuracy(net, data_iter, loss, device):
    """Compute the accuracy for a model on a dataset."""
    net.eval()
    total_loss = 0
    total_hits = 0
    total_samples = 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            total_loss += l.item()
            total_hits += sum(y_hat.argmax(axis=1).type(y.dtype) == y)
            total_samples += y.numel()
    return float(total_loss) / len(data_iter), float(total_hits) / total_samples * 100

def train_epoch(net, train_iter, loss, optimizer, device):
    net.train()
    total_loss = 0
    total_hits = 0
    total_samples = 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        y_hat = net(X)
        l = loss(y_hat, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        total_loss += l.item()
        total_hits += sum(y_hat.argmax(axis=1).type(y.dtype) == y)
        total_samples += y.numel()
    return float(total_loss) / len(train_iter), float(total_hits) / total_samples * 100

def train(net, train_iter, val_iter, test_iter, num_epochs, lr, device):
    """Train a model."""
    train_loss_all = []
    train_acc_all = []
    val_loss_all = []
    val_acc_all = []
    
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    
    net.apply(init_weights)
    print('Training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(net, train_iter, loss, optimizer, device)
        train_loss_all.append(train_loss)
        train_acc_all.append(train_acc)
        val_loss, val_acc = evaluate_accuracy(net, val_iter, loss, device)
        val_loss_all.append(val_loss)
        val_acc_all.append(val_acc)
        print(f'Epoch {epoch + 1}, Train loss {train_loss:.2f}, Train accuracy {train_acc:.2f}, Validation loss {val_loss:.2f}, Validation accuracy {val_acc:.2f}')
    
    test_loss, test_acc = evaluate_accuracy(net, test_iter, loss, device)
    print(f'Test loss {test_loss:.2f}, Test accuracy {test_acc:.2f}')
    
    return train_loss_all, train_acc_all, val_loss_all, val_acc_all

def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

**Plots**

In [ ]:
def plot_loss(train_loss_all, val_loss_all):
    epochs = range(1, len(train_loss_all) + 1)
    plt.plot(epochs, train_loss_all, 'bo', label='Training loss')
    plt.plot(epochs, val_loss_all, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

def plot_accuracy(train_acc_all, val_acc_all):
    epochs = range(1, len(train_acc_all) + 1)
    plt.plot(epochs, train_acc_all, 'bo', label='Training acc')
    plt.plot(epochs, val_acc_all, 'b', label='Validation acc')
    plt.title('Training and validation accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()

**Train the model**

In [ ]:
# Hyperparameters as specified
batch_size = 256
lr = 0.05  # Learning rate: 0.05
num_epochs = 5  # Train for 5 epochs

# Load data
train_iter, val_iter, test_iter = load_data_svhn(batch_size)

# Train the model
train_loss_all, train_acc_all, val_loss_all, val_acc_all = train(
    net, train_iter, val_iter, test_iter, num_epochs, lr, try_gpu()
)

# Plot results
plot_loss(train_loss_all, val_loss_all)
plot_accuracy(train_acc_all, val_acc_all)